In [ ]:
from pathlib import Path

# Find repo root
REPO_ROOT = Path.cwd().parent
print(f"Repo root: {REPO_ROOT}")

REPORT_ROOT = REPO_ROOT / "report"

FIGSIZE = (20,18)
DPI = 100
GENERATE_PLOTS = False

In [ ]:
import pandas as pd
import geopandas as gpd
import numpy as np
import sys
import json
from shapely.geometry import shape
from hotelling.spatial.admin import join_lor_names

# Find repo root
REPO_ROOT = Path.cwd().parent
print(f"Repo root: {REPO_ROOT}")

REPORT_ROOT = REPO_ROOT / "report"

# Add src to path for imports
sys.path.insert(0, str(Path.cwd().parent / 'src'))

from hotelling.spatial.boundaries import load_boundary

PATH_RAW = REPO_ROOT / Path('data/raw')
PATH_PROCESSED = REPO_ROOT / Path('data/processed')

# Midpoint table (center coordinates)
zensus = gpd.read_parquet(PATH_RAW / 'zensus2022_grid.parquet')
zensus_filtered = gpd.read_parquet(PATH_RAW / 'zensus2022_grid_filtered.parquet')
lor = gpd.read_parquet(PATH_PROCESSED / 'lor.parquet')

# CRITICAL FIX: berlin.geojson has EPSG:3035 coordinates but geopandas auto-detects as EPSG:4326
# We must force the correct CRS instead of transforming from the wrong one
with open(PATH_RAW / 'city_boundary_Berlin.geojson', 'r') as f:
    berlin_json = json.load(f)
berlin = gpd.GeoDataFrame([1], geometry=[shape(berlin_json['geometry'])], crs='EPSG:3035')

boundary = load_boundary(PATH_RAW / 'relation_boundary_14983.geojson')

# Load pop_grid

grid = gpd.read_parquet(PATH_PROCESSED / 'pop_grid.parquet')

# Build squares from points of grid
grid['geometry'] = grid.apply(lambda row: row.geometry.buffer(50, cap_style='square'), axis=1)
grid['index'] = grid.index

In [ ]:
# DB Station data
db_stations = pd.read_csv(PATH_RAW / 'db_station_data.csv')
db_stations = db_stations[db_stations['Bundesland'] == 'Berlin'].copy()
display(db_stations)

# Fix station names for matching
db_stations[db_stations['Bahnhof'].str.startswith('lin ')] = db_stations[db_stations['Bahnhof'].str.startswith('lin ')].copy().assign(Bahnhof=lambda df: df['Bahnhof'].str[4:])

In [ ]:
# The OSM stations query has been moved into osm.py as `_STATIONS_TAGS`.
# It fetches all elements tagged railway=station (S-Bahn, U-Bahn, regional,
# and long-distance rail) using `out geom tags;` — equivalent to the original
# query but with inline geometry, so no manual node-ref assembly is needed.
#
# Results are cached to data/raw/OSM_POIs_Berlin_stations.parquet.

from hotelling.spatial.osm import fetch_pois

osm_stations = fetch_pois(type="stations", city="Berlin")
print(f"OSM stations: {len(osm_stations)} rows, {osm_stations.shape[1]} columns")
print(f"Columns: {list(osm_stations.columns)}")
osm_stations.head()


In [ ]:
# Assign the osm_stations to the grid squares and create a column with the count of stations in each square and a column with the list of station names in each square
def assign_stations_to_grid(grid, stations):
    # Spatial join to assign stations to grid squares
    joined = gpd.sjoin(stations, grid, how='left', predicate='within')
    
    # Count stations in each square
    station_counts = joined.groupby('index').size().rename('station_count')
    
    # List of station names in each square
    station_names = joined.groupby('index')['name'].apply(list).rename('station_names')
    
    # Merge counts and names back to the grid
    grid = grid.merge(station_counts, left_on='index', right_index=True, how='left')
    grid = grid.merge(station_names, left_on='index', right_index=True, how='left')
    
    # Fill NaN values with 0 for counts and empty list for names
    grid['station_count'] = grid['station_count'].fillna(0).astype(int)
    grid['station_names'] = grid['station_names'].apply(lambda x: x if isinstance(x, list) else [])
    
    return grid

grid_with_stations = assign_stations_to_grid(grid, osm_stations.to_crs(grid.crs))
grid_with_stations.head()

grid_with_stations['matched_db_stations'] = np.nan  # Initialize with NaN


In [ ]:
osm_stations

In [ ]:
import re
import unicodedata

def norm(s: str) -> str:
    s = unicodedata.normalize("NFKD", s)
    s = "".join(c for c in s if not unicodedata.combining(c))
    s = s.lower()
    s = re.sub(r"\(.*?\)", "", s)          # drop parenthetical suffixes
    s = re.sub(r"^(s|u)\s+", "", s)        # drop S/U prefixes
    s = re.sub(r"^berlin[\s-]+", "", s)    # drop Berlin- / Berlin 
    s = s.replace("ß", "ss")
    s = re.sub(r"[^a-z0-9]+", " ", s)
    return re.sub(r"\s+", " ", s).strip()

# First match by normalized name
db_lookup = {}
for name in db_stations['Bahnhof'].unique():
    db_lookup.setdefault(norm(name), name)

# Manual overrides for the non-exact cases where DB has a different station name
manual = {
    "Schöneweide": "Berlin-Schöneweide Pbf",
    "Berlin-Schöneweide": "Berlin-Schöneweide Pbf",
    "Wittenau": "Berlin-Wittenau (Wilhelmsruher Damm)",
    # Uncomment only if you want best-effort guesses instead of strict matching:
    # "Friedrichsfelde": "Friedrichsfelde Ost",
    # "Biesdorf-Süd": "Biesdorf",
}

osm_to_db = {
    name: manual.get(name, db_lookup.get(norm(name)))
    for name in osm_stations['name'].unique()
}

In [ ]:
for osm_name, db_name in osm_to_db.items():
    if db_name is None:
        print(f"No match for OSM station '{osm_name}' with {"BVG" if osm_stations[osm_stations['name'] == osm_name]['operator'].values[0] == "Berliner Verkehrsbetriebe" else osm_stations[osm_stations['name'] == osm_name]['operator'].values[0]}")
        

In [ ]:
'''
from fuzzywuzzy import fuzz

# Create a list of station names from the OSM data
osm_station_names = osm_stations['official_name'].tolist()

# Create a list of station names from the DB data
db_station_names = db_stations['Bahnhof'].tolist()

cells_with_stations = grid_with_stations[grid_with_stations['station_count'] > 0]

matches = {"idx": [], "osm_names": [], "db_name": [], "score": []}
for idx, row in cells_with_stations.iterrows():
    cell_station_names = row['station_names']
    best_match = None
    best_score = 0
    for db_name in db_station_names:
        single_best_match = None
        single_best_score = 0
        for osm_name in cell_station_names:
            score = fuzz.ratio(db_name, osm_name)
            if score > single_best_score:
                single_best_score = score
                single_best_match = db_name
        if single_best_score > best_score:
            best_score = single_best_score
            best_match = single_best_match
    matches["idx"].append(idx)
    matches["osm_names"].append(cell_station_names)
    matches["db_name"].append(best_match)
    matches["score"].append(best_score)
    
# Print the matches
matches_df = pd.DataFrame(matches)
display(matches_df)
'''

In [ ]:
for idx, row in grid_with_stations.iterrows():
    if row['station_count'] > 0:
        matched_db_stations = []
        db_names_in_cell = set()
        for osm_name in row['station_names']:
            db_name = osm_to_db.get(osm_name)
            if db_name:
                matched_db_stations.append(db_name)
                db_names_in_cell.add(db_name)
        # Check for duplicates
        if len(db_names_in_cell) < len(matched_db_stations):
            print(f"Warning: Duplicate DB station matches in cell {idx} for OSM stations {row['station_names']}. Matched DB stations: {matched_db_stations}")
        
        if len(db_names_in_cell) == 1:
            grid_with_stations.at[idx, 'matched_db_stations'] = list(db_names_in_cell)[0]
        elif len(db_names_in_cell) == 0:
            print(f"No DB station match in cell {idx} for OSM stations {row['station_names']}")
        else:
            print(f"Multiple different DB station matches in cell {idx} for OSM stations {row['station_names']}.")

In [ ]:
grid_with_stations['station_class'] = np.nan  # Initialize with NaN

for idx, row in grid_with_stations.iterrows():
    if row['station_count'] > 0 and not pd.isna(row['matched_db_stations']):
        db_name = row['matched_db_stations']
        db_info = db_stations[db_stations['Bahnhof'] == db_name]
        if not db_info.empty:
            station_class = db_info['klasse'].values[0]
            grid_with_stations.at[idx, 'station_class'] = int(station_class)
        else:
            print(f"DB station '{db_name}' not found in DB stations data.")
        

In [ ]:
import matplotlib.pyplot as plt
import contextily as ctx

if GENERATE_PLOTS:
    # Plot the grid colored by station class
    fig, ax = plt.subplots(figsize=FIGSIZE, dpi=DPI)
    grid_with_stations.plot(column='station_class', ax=ax, cmap='tab10', edgecolor='none', legend=True, alpha=1)
    grid_with_stations.plot(ax=ax, color = 'firebrick', linewidth=1, alpha=0.2)
    berlin.plot(ax=ax, edgecolor='royalblue', color='none', linewidth=2)
    ctx.add_basemap(ax, crs=berlin.crs, source=ctx.providers.OpenStreetMap.Mapnik, zoom=11, zorder=0, alpha = 0.3)
    ax.set_axis_off()
    plt.title("Grid with DB Station Classes")
    plt.show()


In [ ]:
# Save the grid with station info to a new Parquet file
grid_with_stations.to_parquet(PATH_PROCESSED / 'grid_with_stations.parquet')

In [ ]:
if not GENERATE_PLOTS:
    import nbformat, pathlib

    _nb_path = pathlib.Path(__file__) if "__file__" in dir() else None
    # Fallback: set explicitly if auto-detection unavailable
    _nb_path = pathlib.Path("GEO_04_transport_hubs.ipynb")  # ← set once per notebook

    _nb = nbformat.read(_nb_path, as_version=4)
    for _cell in _nb.cells:
        _cell["outputs"] = []
        _cell["execution_count"] = None
    nbformat.write(_nb, _nb_path)
    print(f"Outputs cleared: {_nb_path.name}")